In [ ]:
import pandas as pd
import numpy as np

# -------- CONFIG --------
INFILE  = "/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Fundamentals/Data/csv files/own code/apple_financials_2009_2025.csv"     # <-- change if needed
OUTFILE = "apple_financials_2009_2025_clean_quarters.csv"

FLOW_COLS    = ["revenue", "net_income"]                  # period flows
INSTANT_COLS = ["total_assets", "total_liabilities", "shareholders_equity"]  # end-of-period

# -------- LOAD --------
financials_09_25 = pd.read_csv(INFILE, parse_dates=["filing_date", "period_end"])
for c in FLOW_COLS + INSTANT_COLS:
    if c in financials_09_25.columns:
        financials_09_25[c] = pd.to_numeric(financials_09_25[c], errors="coerce")

# -------- HELPERS --------
def is_valid_period_end(d: pd.Timestamp) -> bool:
    """
    Apple uses a 52/53-week calendar; quarter ends are Saturdays near:
      - Q1: last Sat of Dec
      - Q2: last Sat of Mar OR first Sat of Apr
      - Q3: last Sat of Jun OR first Sat of Jul
      - Q4: last Sat of Sep OR first Sat of Oct (rare 53-week)
    """
    if pd.isna(d): return False
    if d.weekday() != 5:  # Saturday
        return False
    m, day = d.month, d.day
    if m == 12 and 24 <= day <= 31: return True
    if (m == 3 and day >= 24) or (m == 4 and day <= 7): return True
    if (m == 6 and day >= 24) or (m == 7 and day <= 7): return True
    if (m == 9 and day >= 24) or (m == 10 and day <= 7): return True
    return False

def fiscal_year_from_period_end(d: pd.Timestamp) -> int:
    # Dec belongs to next FY; others use same year
    if d.month == 12:
        return d.year + 1
    return d.year

def quarter_label_from_period_end(d: pd.Timestamp) -> str:
    # Apple fiscal: Q1 ends in late Dec → FY+1; Q2 late Mar/early Apr; Q3 late Jun/early Jul; Q4 late Sep/early Oct
    if d.month == 12: return f"Q1 {d.year + 1}"
    if d.month in (3, 4): return f"Q2 {d.year}"
    if d.month in (6, 7): return f"Q3 {d.year}"
    if d.month in (9, 10): return f"Q4 {d.year}"
    return None

def fq_num_from_label(q: str):
    if isinstance(q, str) and q.startswith("Q"):
        try: return float(q[1])
        except: return np.nan
    return np.nan

# -------- 1) DEDUP: keep latest filing per (form, period_end) --------
financials_09_25 = (
    financials_09_25
    .sort_values(["period_end", "form", "filing_date"])     # earliest .. latest
    .groupby(["period_end", "form"], as_index=False)
    .last()                                                 # keep latest filing
)

# -------- 2) DROP obvious spurious period_end --------
financials_09_25 = financials_09_25[financials_09_25["period_end"].apply(is_valid_period_end)].copy()

# Also: keep only September/early-Oct 10-Ks (true annuals)
financials_09_25 = financials_09_25[~((financials_09_25["form"] == "10-K") &
                                      ~(financials_09_25["period_end"].dt.month.isin([9, 10])) )].copy()

# -------- 3) ASSIGN fiscal labels --------
financials_09_25["fy"]      = financials_09_25["period_end"].apply(fiscal_year_from_period_end).astype("Int64")
financials_09_25["quarter"] = financials_09_25["period_end"].apply(quarter_label_from_period_end)
financials_09_25["fq"]      = financials_09_25["quarter"].map(fq_num_from_label)

# -------- 4) SYNTHESIZE Q4 if missing (flows from 10-K - (Q1+Q2+Q3); instants from 10-K) --------
q = financials_09_25[financials_09_25["form"] == "10-Q"].copy()
k = financials_09_25[financials_09_25["form"] == "10-K"].copy()

# Sum flows for Q1..Q3 per FY
sum_q1_3 = (q[q["fq"].isin([1.0, 2.0, 3.0])]
            .groupby("fy")[FLOW_COLS]
            .sum(min_count=1))

# Annual flows & instants per FY from 10-K (one per FY after dedup)
annual = (k.sort_values("period_end")
            .drop_duplicates(subset=["fy"], keep="last")
            .set_index("fy"))

annual_flows    = annual[FLOW_COLS] if set(FLOW_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_instants = annual[INSTANT_COLS] if set(INSTANT_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_meta     = annual[["filing_date", "period_end"]] if {"filing_date","period_end"}.issubset(annual.columns) else pd.DataFrame(index=annual.index)

# Compute Q4 flows
q4_flows = (annual_flows - sum_q1_3).dropna(how="all")

# Which FYs still lack a Q4 10-Q?
have_q4 = set(q[q["fq"] == 4.0]["fy"].dropna().astype(int).tolist())
need_q4 = [fy for fy in q4_flows.index if fy not in have_q4]

q4_rows = []
for fy in need_q4:
    p_end  = annual_meta.loc[fy, "period_end"] if "period_end" in annual_meta.columns else pd.Timestamp(f"{fy}-09-30")
    fdate  = annual_meta.loc[fy, "filing_date"] if "filing_date" in annual_meta.columns else p_end

    row = {
        "filing_date": fdate,
        "period_end":  p_end,
        "form":        "10-Q",
        "fy":          int(fy),
        "quarter":     f"Q4 {fy}",
        "fq":          4.0,
        "synthetic":   True,
        "note":        "Q4 flows = 10K - (Q1+Q2+Q3); instants from 10K",
    }
    for c in FLOW_COLS:
        row[c] = q4_flows.loc[fy, c] if c in q4_flows.columns else np.nan
    for c in INSTANT_COLS:
        row[c] = annual_instants.loc[fy, c] if c in annual_instants.columns else np.nan
    q4_rows.append(row)

q4_df = pd.DataFrame(q4_rows)

# Mark originals
financials_09_25["synthetic"] = False
financials_09_25["note"] = ""

# Combine and sort
full = pd.concat([financials_09_25, q4_df], ignore_index=True)
full["filing_date"] = pd.to_datetime(full["filing_date"])
full = full.sort_values(["fy", "period_end", "form", "filing_date"]).reset_index(drop=True)

# === 5) AÑADIR FILAS EN BLANCO SI FALTAN TRIMESTRES (Q1..Q4) POR FY ===
# Trabajamos sobre vista trimestral (10-Q o sintéticos) para evitar duplicados por 10-K
quarterly_view = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

# Extrae 'FY' y el código de trimestre (Q1..Q4) a partir de 'quarter'
quarterly_view["fy"] = quarterly_view["quarter"].str.extract(r"\b(\d{4})\b").astype("Int64")
quarterly_view["qcode"] = quarterly_view["quarter"].str.extract(r"(Q[1-4])")

# Si hay más de una fila por (fy, q) (p.ej., múltiples filings),
# prioriza 10-Q frente a 10-K (por si se coló alguno), y la última filing_date
priority = {"10-Q": 0, "10-K": 1}
quarterly_view["form_priority"] = quarterly_view["form"].map(priority).fillna(2).astype(int)

quarterly_view = (
    quarterly_view.sort_values(["fy", "qcode", "form_priority", "filing_date"])
                  .groupby(["fy", "qcode"], as_index=False)
                  .last()
)

# Construye el MultiIndex completo FY x (Q1..Q4)
fy_min, fy_max = int(quarterly_view["fy"].min()), int(quarterly_view["fy"].max())
full_index = pd.MultiIndex.from_product(
    [range(fy_min, fy_max + 1), ["Q1", "Q2", "Q3", "Q4"]],
    names=["fy", "qcode"]
)

# Reindexa para introducir filas en blanco donde falten trimestres
quarterly_full = (
    quarterly_view.set_index(["fy", "qcode"])
                  .reindex(full_index)
                  .reset_index()
)

# Reconstruye la etiqueta 'quarter' (p.ej., "Q1 2014")
quarterly_full["quarter"] = quarterly_full["qcode"] + " " + quarterly_full["fy"].astype("Int64").astype(str)

# Integra de vuelta con el DataFrame 'full':
#  - Eliminamos posibles duplicados por 'quarter' en 'full'
#  - Usamos 'quarterly_full' como base trimestral definitiva (con NaNs donde faltaba)
cols_to_keep = full.columns.tolist()
# Asegura que 'quarterly_full' tenga todas las columnas (las ausentes se crean como NaN)
for c in cols_to_keep:
    if c not in quarterly_full.columns:
        quarterly_full[c] = np.nan

# Nos quedamos con las columnas en el mismo orden
full = quarterly_full[cols_to_keep]

# (Optional) Keep purely quarterly view (drop 10-Ks) once Q4 is synthesized:
# full = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

# Quick check: quarters present per FY
check = (full.assign(fq_lbl=lambda d: d["quarter"].str.extract(r"(Q[1-4])"))
              .pivot_table(index="fy", columns="fq_lbl", values="period_end", aggfunc="count")
              .fillna(0).astype(int))
print("Quarters present per FY:\n", check)

# Drop unecessary columns

full = full.drop(columns=["filing_date", "fy", "fq"])

# Save
full.to_csv(OUTFILE, index=False)
print(f"[DONE] Saved cleaned dataset → {OUTFILE}")


Quarters present per FY:
 fq_lbl  Q1  Q2  Q3  Q4
fy                    
2009     0   0   1   1
2010     1   1   1   1
2011     1   1   1   1
2012     1   1   1   1
2013     1   1   1   1
2014     1   1   1   1
2015     0   1   0   1
2016     0   0   0   0
2017     1   1   1   0
2018     0   1   1   1
2019     1   1   1   1
2020     1   1   1   0
2021     1   1   1   1
2022     1   1   1   1
2023     1   1   1   1
2024     1   1   1   1
2025     1   1   1   0
[DONE] Saved cleaned dataset → apple_financials_2009_2025_clean_quarters_v2.csv
